# KohakuMoss on Google Colab

Train a LoRA / LyCORIS adapter for MOSS-Image on a Colab GPU runtime.
Runtime > Change runtime type: **RTX PRO 6000 (G4)** or **A100**.
Install details: `docs/guides/install.md`; the config presets: `docs/guides/configs.md`.

Outputs (checkpoints, samples, exports) go to Google Drive, so they survive a
disconnect; the venv, the weights and the caches live on the VM disk.

In [ ]:
# @title Settings
REPO = "https://github.com/KohakuBlueleaf/KohakuMossTrainer"  # @param {type:"string"}
WEIGHTS_REPO = "OpenMOSS-Team/MOSS-Image-Base"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/kohakumoss"  # @param {type:"string"}
DATASET = "/content/drive/MyDrive/kohakumoss/datasets/my_dataset"  # @param {type:"string"}
DATASET_SOURCE = "kohya"  # @param ["kohya", "folder", "folder_nl", "jsonl", "parquet", "webdataset", "moss_shard"]
ALGO = "lora"  # @param ["lora", "loha", "lokr", "oft", "dora", "full"]
RUN_NAME = "my-lora"  # @param {type:"string"}
STEPS = 2000  # @param {type:"integer"}
SAMPLE_PROMPT = "a lighthouse on a rocky coast at sunset"  # @param {type:"string"}

KM = "/content/KohakuMossTrainer"
VENV = "/content/km-venv"
WEIGHTS = "/content/weights/MOSS-Image"
DATA = "/content/data/" + DATASET.rstrip("/").split("/")[-1]

In [ ]:
# @title Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p {DRIVE_ROOT}/runs {DRIVE_ROOT}/kernel-cache

In [ ]:
# @title Install (venv with the torch build for this GPU's driver, then the doctor report)
!git clone --depth 1 {REPO} {KM} 2>/dev/null || git -C {KM} pull --ff-only
!python {KM}/scripts/colab/setup.py --venv {VENV}

In [ ]:
# @title Download the weights to the VM disk (~26 GB)
!{VENV}/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('{WEIGHTS_REPO}', local_dir='{WEIGHTS}/base', max_workers=16)"

In [ ]:
# @title Copy the dataset from Drive to the VM disk
!mkdir -p /content/data && rsync -a --info=progress2 {DATASET}/ {DATA}/

In [ ]:
# @title Write the run config (presets for this GPU, then the Colab paths)
!cd {KM} && {VENV}/bin/kohakumoss-config new train --weights {WEIGHTS}/base --data {DATA} --source {DATASET_SOURCE} --algo {ALGO} --name {RUN_NAME} --steps {STEPS} --prompt "{SAMPLE_PROMPT}" --out /content/run_base.py --force

config = f'''"""Colab run: the generated starter with online caches on the VM disk and outputs
on Drive. Drives scripts/train/moss.py."""

from kohakuengine import use_config

use_config("/content/run_base.py")

LATENT_CACHE = "online"
TEXT_CACHE = "online"
CACHE_DIR = "/content/cache"
OUTPUT_DIR = "{DRIVE_ROOT}/runs"
'''
with open("/content/run_config.py", "w") as f:
    f.write(config)
print(config)

In [ ]:
# @title Train
%cd {KM}
!KOHAKUMOSS_KERNEL_CACHE={DRIVE_ROOT}/kernel-cache {VENV}/bin/kogine run scripts/train/moss.py --config /content/run_config.py

In [ ]:
# @title Sample with the exported adapter (full finetune: the merged transformer)
import glob

export = f"{DRIVE_ROOT}/runs/{RUN_NAME}/export"
if ALGO == "full":
    merged = sorted(glob.glob(f"{export}/*-merged.safetensors"))[-1]
    adapter_flag, extra = "", f"--set TRANSFORMER={merged}"
else:
    adapter = sorted(glob.glob(f"{export}/*-lycoris.safetensors"))[-1]
    adapter_flag, extra = f"--adapter {adapter}", ""
!cd {KM} && {VENV}/bin/kohakumoss-config new sample --weights {WEIGHTS}/base {adapter_flag} --prompt "{SAMPLE_PROMPT}" --samples {DRIVE_ROOT}/runs/{RUN_NAME}/sample_cli --out /content/sample_config.py --force
!cd {KM} && {VENV}/bin/kogine run scripts/sample/sample.py --config /content/sample_config.py --set STEPS=30 {extra}

In [ ]:
# @title Release the runtime (stops compute-unit billing)
from google.colab import runtime

runtime.unassign()